# Gold Output Validation

Lightweight validation notebook for the generated Gold v1 job-market artifacts.

Pipeline logic belongs under `src/`; this notebook only inspects the outputs.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
if (cwd / "data" / "gold").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data" / "gold").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "02-job-market-analytics" / "data" / "gold").exists():
    PROJECT_ROOT = cwd / "projects" / "02-job-market-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

gold_dir = PROJECT_ROOT / "data" / "gold"
gold_files = {
    "job_title_summary": gold_dir / "job_title_summary.csv",
    "industry_summary": gold_dir / "industry_summary.csv",
    "location_summary": gold_dir / "location_summary.csv",
    "automation_ai_summary": gold_dir / "automation_ai_summary.csv",
}
gold_files

In [ ]:
outputs = {name: pd.read_csv(path) for name, path in gold_files.items()}
{name: df.shape for name, df in outputs.items()}

## Output Previews

In [ ]:
for name, df in outputs.items():
    print(name)
    display(df.head())
    print(df.columns.tolist())
    print()

## Sanity Checks

In [ ]:
{name: int(df["total_records"].sum()) for name, df in outputs.items()}

In [ ]:
share_check = {}
for name, df in outputs.items():
    share_columns = [column for column in df.columns if column.endswith("_share")]
    share_check[name] = {
        column: bool(df[column].between(0, 1).all())
        for column in share_columns
    }
share_check

In [ ]:
salary_check = {}
for name, df in outputs.items():
    salary_columns = [column for column in df.columns if column.endswith("salary_usd")]
    salary_check[name] = {
        column: bool((df[column] >= 0).all())
        for column in salary_columns
    }
salary_check

## Quick Observations

- Each Gold output has a documented grain and can be translated into a future DBT mart.
- Share metrics should stay between 0 and 1.
- Salary metrics should remain non-negative.
- The grouped `total_records` sums are not meant to be added across all outputs together; each output summarizes the same Silver source at a different grain.